# VoxCPM2 - clone giọng tiếng Việt trên Kaggle (GPU T4 miễn phí)

**Trước khi chạy:**

1. `Settings` -> `Accelerator` -> chọn **GPU T4 x2**
2. `Settings` -> **internet đang BẬT** (menu ghi "Turn off internet" = đang bật, đúng rồi)
3. Panel phải -> `Input` -> nút **`Upload`** -> **`New Dataset`** -> tải lên `ai-dev-voice-clean.wav`

Rồi bấm `Run All`. Lần đầu tải model ~5GB.

---
### Bài học quan trọng

VoxCPM2 có 2 chế độ clone, khác nhau rất nhiều:

| Chế độ | Cách gọi | Đặc điểm |
|---|---|---|
| **Ultimate** | `prompt_wav_path` + `prompt_text` + `reference_wav_path` | Giống giọng nhất, NHƯNG sao chép luôn **nhịp nói** của mẫu và **TẮT điều khiển phong cách** |
| **Cơ bản** | chỉ `reference_wav_path` | Cho phép **điều khiển phong cách** bằng chỉ thị `(...)` ở đầu văn bản |

Notebook này dùng **chế độ cơ bản**. Nếu giọng mẫu nói nhanh, chế độ Ultimate sẽ ép bài
đọc nhanh theo và không cách nào chỉnh được.

## Ô 1 - Cài đặt

In [ ]:
!pip install -q --no-deps voxcpm==2.0.3
!pip install -q einops librosa soundfile hf_transfer
print('Cai xong.')

## Ô 2 - Tải model + nạp lên GPU

Ô này tự giải phóng model cũ và tự chọn GPU còn trống - chạy lại bao nhiêu lần cũng không tràn VRAM.

In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

import json, glob, gc, torch, numpy as np, soundfile as sf, librosa
from huggingface_hub import snapshot_download

# --- Giai phong model cu (neu chay lai o nay) ---
if 'model' in globals():
    del model
gc.collect(); torch.cuda.empty_cache()

# --- Tu chon GPU con trong nhat (T4 x2: neu GPU0 dang ban thi dung GPU1) ---
best, best_free = 0, -1
for gi in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(gi)
    print(f'  GPU{gi}: trong {free/1e9:.1f}/{total/1e9:.1f} GB')
    if free > best_free:
        best, best_free = gi, free
DEVICE = f'cuda:{best}'
assert best_free > 6e9, 'Ca 2 GPU deu day. Vao menu Run -> Restart Session roi chay lai.'

cc = torch.cuda.get_device_capability(best)
print(f'Dung {DEVICE} ({torch.cuda.get_device_name(best)}, sm_{cc[0]}{cc[1]}) | torch {torch.__version__}')

local_dir = snapshot_download('openbmb/VoxCPM2')

# T4 = sm_75, khong ho tro bf16 -> ep float16.
cfg_path = os.path.join(local_dir, 'config.json')
cfg = json.load(open(cfg_path, encoding='utf-8'))
if cc[0] < 8 and cfg.get('dtype') == 'bfloat16':
    cfg['dtype'] = 'float16'
    json.dump(cfg, open(cfg_path, 'w', encoding='utf-8'))
print('dtype:', cfg.get('dtype'))

from voxcpm import VoxCPM
model = VoxCPM.from_pretrained(local_dir, load_denoiser=False, optimize=False, device=DEVICE)
SR = model.tts_model.sample_rate
print('San sang. Sample rate:', SR)

## Ô 3 - Giọng mẫu + văn bản

Dùng **nguyên file giọng mẫu**, không xử lý gì (kéo giãn làm giọng bị méo).
`REF_TEXT` phải gõ **đúng từng chữ** lời trong file mẫu.


In [ ]:
cands = sorted(glob.glob('/kaggle/input/**/*.wav', recursive=True) +
               glob.glob('/kaggle/input/**/*.mp3', recursive=True) +
               glob.glob('/kaggle/input/**/*.MP3', recursive=True))
assert cands, 'Chua thay file giong mau. Panel phai -> Input -> Upload -> New Dataset'
REF_WAV = cands[0]          # DUNG NGUYEN file mau, khong xu ly gi
print('Giong mau:', REF_WAV)

# LOI MAU: go DUNG TUNG CHU loi thoai trong file mau.
REF_TEXT = ("Mà khoan đã, đời đâu có dễ vậy. Mà muốn đối chiếu, thì phải dựa trên dữ liệu. "
            "Mấy mô hình học trên cả núi dữ liệu, mỗi bước mà bắt nó dò lại hết cả núi đó, "
            "thì chạy tới mùa quýt mới xong, vậy ngoài thực tế người ta làm sao cho kịp đây")

TEXT = """Trên đường đạo cũng giống như trên đường đời, sẽ có nhiều thử thách. Nếu vừa gặp chút thử thách đã vội ngã lòng, mất đạo tâm, thì không thể tiến xa được. Những thử thách, những nghịch cảnh, những trở ngại, những ngang trái, những oan ức, đều là cơ hội rèn luyện đạo tâm của ta, hãy dùng những thử thách đó như món quà quý giá chứ đừng quay lưng chạy trốn."""

print(f'{len(TEXT)} ky tu')


---
## Chỉnh cho vừa tai

Ô 4 in ra **tốc độ ký tự/giây** - dùng con số đó để chỉnh, đừng mò:

| Muốn | Chỉnh |
|---|---|
| **Đọc chậm hơn** | Giảm `SLOW_REF` ở Ô 3 (0.72 -> 0.65). Cần gạt mạnh nhất |
| Đọc nhanh hơn | Tăng `SLOW_REF` (0.85) hoặc đặt `0` để dùng mẫu gốc |
| Ngắt nghỉ lâu hơn | Tăng `GAP_PARA` / `GAP_SENT` ở Ô 4 |
| Ngữ điệu khác | Đổi `STYLE` (tiếng Anh/Trung, ngắn gọn) |
| Câu dài đọc liền hơi | Giảm `MAX_CHARS` xuống 200 |
| Bám văn bản hơn | `cfg_value` 2.0 -> 2.5 hoặc 3.0 |
| Mượt hơn (chậm hơn) | `inference_timesteps` 10 -> 20 |

Model sinh **ngẫu nhiên** - chạy lại Ô 4 vài lần rồi chọn bản ưng nhất.

## Những thứ KHÔNG tồn tại (đừng tìm)

Không có tham số `speed` / `rate` / `tempo` / `duration`; không hỗ trợ SSML `<break>`;
không có token ngắt nghỉ. Xuống dòng trong văn bản **bị xóa** (mã nguồn: `text.replace("\n", " ")`).
Cách duy nhất đổi nhịp đọc là **đổi nhịp của giọng mẫu** (`SLOW_REF`).

## Lỗi thường gặp

| Triệu chứng | Xử lý |
|---|---|
| `CUDA out of memory` khi nạp model | Ô 2 đã tự dọn + tự đổi GPU. Nếu vẫn lỗi: `Run` -> `Restart Session` |
| `CUDA out of memory` khi sinh | Giảm `MAX_CHARS` xuống 200, hoặc `inference_timesteps` xuống 6 |
| Nó **đọc to câu tiếng Anh** trong ngoặc | Bạn đang ở chế độ Ultimate. Bỏ `prompt_wav_path`/`prompt_text` |
| Audio im lặng / rè toàn bộ | fp16 tràn số. Ô 2 đổi `cfg['dtype']` thành `'float32'`, chạy lại từ Ô 2 |
| Giọng bớt giống bản gốc | Đánh đổi của chế độ cơ bản. Muốn giống tối đa: thu giọng mẫu mới đọc chậm rồi quay lại Ultimate |

**Giới hạn Kaggle:** 30 **giờ** GPU/tuần (không phải 30 phút), 12 giờ/phiên,
reset thứ Bảy 00:00 UTC. Xem tại `kaggle.com/me/account`.
Đồng hồ `Session 31m / 12 hours` trên thanh công cụ là thời gian **phiên hiện tại**, không phải quota tuần.

---
## Ô cuối - BẢNG ĐIỀU KHIỂN (thêm nhấn nhá)

Chạy Ô 1, 2, 3 xong rồi chạy ô này. Có 3 chế độ:
- **Ultimate**: giống giọng nhất, nhưng copy ngữ điệu từ mẫu (mẫu phẳng -> ra phẳng).
- **Ghép 2 giọng**: giữ chất giọng của bạn, lấy **ngữ điệu** từ một clip đọc diễn cảm
  (phải upload thêm clip đó + gõ đúng lời của nó).
- **Style**: giữ chất giọng + chỉ thị biểu cảm tiếng Anh trong `(...)`.

Thử nhanh không cần clip mới: hạ **cfg_value** về 1.3-1.6.


In [ ]:
import ipywidgets as W
from IPython.display import display, Audio, clear_output
import re, time, inspect, glob, numpy as np, soundfile as sf

# ============================================================================
# 4 CACH THEM NHAN NHA / NGU DIEU (doc tu ma nguon VoxCPM2):
#   Ultimate  = giong giong nhat, nhung SAO CHEP ngu dieu tu mau -> mau phang thi ra phang.
#   Ghep 2 giong = chat giong tu clip cua ban (reference) + NGU DIEU tu clip dien cam (prompt).
#                  Hai khe nay doc lap trong kien truc. Can upload 1 clip dien cam + loi cua no.
#   Style     = chi dung chat giong + CHI THI bieu cam trong (...) (tieng Anh). Bo Ultimate.
#   cfg_value thap (~1.5) = model tu do bien tau hon (bot phang). Ap cho ca 3 che do.
# ============================================================================

# Liet ke moi file audio da upload (dung cho khe "ngu dieu" o che do Ghep 2 giong)
_audio = sorted(glob.glob('/kaggle/input/**/*.wav', recursive=True) +
                glob.glob('/kaggle/input/**/*.mp3', recursive=True) +
                glob.glob('/kaggle/input/**/*.MP3', recursive=True))

_LO = W.Layout(width='620px')
_ST = {'description_width': '135px'}

w_mode = W.RadioButtons(
    options=[('Ultimate - giọng giống nhất (đều đều nếu mẫu phẳng)', 'ultimate'),
             ('Ghép 2 giọng - chất giọng của bạn + ngữ điệu từ clip diễn cảm', 'twoslot'),
             ('Style - giọng của bạn + chỉ thị biểu cảm (bỏ Ultimate)', 'style')],
    value='ultimate', description='Chế độ:', layout=W.Layout(width='700px'), style=_ST)

w_dir = W.Text(value='(expressive, emphatic delivery, dynamic rising and falling intonation)',
               description='Chỉ thị (Style):', layout=_LO, style=_ST)
w_pros_file = W.Dropdown(options=(_audio or ['(chưa có file)']),
                         value=(_audio[-1] if _audio else '(chưa có file)'),
                         description='Clip ngữ điệu:', layout=_LO, style=_ST)
w_pros_text = W.Textarea(value='', placeholder='Gõ ĐÚNG TỪNG CHỮ lời thoại trong clip ngữ điệu...',
                         description='Lời clip ngữ điệu:', layout=W.Layout(width='620px', height='70px'), style=_ST)

w_cfg   = W.FloatSlider(value=1.5, min=1.0, max=3.0, step=0.1, description='cfg_value:',
                        readout_format='.1f', continuous_update=False, layout=_LO, style=_ST)
w_steps = W.IntSlider(value=12, min=4, max=30, step=1, description='Số bước:',
                      continuous_update=False, layout=_LO, style=_ST)
w_max   = W.IntSlider(value=200, min=60, max=400, step=10, description='Ký tự/cụm:',
                      continuous_update=False, layout=_LO, style=_ST)
w_gap_s = W.FloatSlider(value=0.45, min=0.0, max=1.5, step=0.05, description='Nghỉ hết câu:',
                        readout_format='.2f', continuous_update=False, layout=_LO, style=_ST)
w_gap_p = W.FloatSlider(value=0.75, min=0.0, max=3.0, step=0.05, description='Nghỉ hết đoạn:',
                        readout_format='.2f', continuous_update=False, layout=_LO, style=_ST)
w_text  = W.Textarea(value=TEXT, description='Văn bản:', layout=W.Layout(width='880px', height='150px'), style=_ST)
w_go    = W.Button(description='ĐỌC', button_style='primary', icon='play',
                   layout=W.Layout(width='150px', height='42px'))
w_out   = W.Output()


def _cum(text, limit):
    out = []
    for para in [p for p in text.replace(chr(13), '').split(chr(10)) if p.strip()]:
        cau = [s.strip() for s in re.split(r'(?<=[.!?…])\s+', para) if s.strip()]
        buf = ''
        for s in cau:
            if buf and len(buf) + len(s) + 1 > limit:
                out.append((buf, 'sent')); buf = s
            else:
                buf = (buf + ' ' + s).strip()
        if buf:
            out.append((buf, 'para'))
    return out


def _kwargs_for(mode, s):
    """Xay tham so goi model.generate theo che do."""
    if mode == 'twoslot':
        return dict(text=s, reference_wav_path=REF_WAV,          # chat giong cua ban
                    prompt_wav_path=w_pros_file.value,           # ngu dieu tu clip dien cam
                    prompt_text=w_pros_text.value.strip())
    if mode == 'style':
        return dict(text=w_dir.value.strip() + ' ' + s, reference_wav_path=REF_WAV)
    return dict(text=s, prompt_wav_path=REF_WAV, prompt_text=REF_TEXT, reference_wav_path=REF_WAV)


def _doc(_):
    with w_out:
        clear_output(wait=True)
        txt = w_text.value.strip()
        if not txt:
            print('Chưa nhập văn bản.'); return
        mode = w_mode.value
        if mode == 'twoslot':
            if not _audio or w_pros_file.value not in _audio:
                print('Chế độ Ghép 2 giọng cần UPLOAD thêm 1 clip diễn cảm (Input -> Upload -> New Dataset), '
                      'rồi chọn ở "Clip ngữ điệu".'); return
            if not w_pros_text.value.strip():
                print('Hãy gõ "Lời clip ngữ điệu" - đúng từng chữ lời thoại trong clip diễn cảm.'); return
        w_go.disabled = True; w_go.description = 'Đang đọc...'
        try:
            ok = set(inspect.signature(model._generate).parameters)
            gk = {k: v for k, v in dict(cfg_value=w_cfg.value, inference_timesteps=w_steps.value,
                                        normalize=False).items() if k in ok}
            cum = _cum(txt, w_max.value)
            print(f'{len(txt)} ký tự -> {len(cum)} cụm | chế độ: {mode} | cfg {w_cfg.value:.1f}')
            parts, t0 = [], time.time()
            for i, (s, kind) in enumerate(cum):
                wav = np.asarray(model.generate(**_kwargs_for(mode, s), **gk), dtype=np.float32)
                parts.append(wav)
                if i < len(cum) - 1:
                    gap = w_gap_p.value if kind == 'para' else w_gap_s.value
                    if gap > 0:
                        parts.append(np.zeros(int(gap * SR), dtype=np.float32))
                print(f'  cụm {i+1}/{len(cum)}: {len(s)} ký tự -> {len(wav)/SR:.1f}s')
            audio = np.concatenate(parts)
            peak = float(np.max(np.abs(audio)))
            if peak > 0.99:
                audio = audio * (0.99 / peak)
            out_path = '/kaggle/working/voxcpm2_output.wav'
            sf.write(out_path, audio, SR)
            print(f'\nXong sau {time.time()-t0:.0f}s | {len(audio)/SR:.1f}s')
            display(Audio(out_path, autoplay=False))
            print(f'Tải về: panel phải -> Output -> {out_path}')
        except Exception as e:
            print(f'LỖI {type(e).__name__}: {e}')
        finally:
            w_go.disabled = False; w_go.description = 'ĐỌC'


def _toggle(*_):
    m = w_mode.value
    w_dir.layout.display = '' if m == 'style' else 'none'
    w_pros_file.layout.display = '' if m == 'twoslot' else 'none'
    w_pros_text.layout.display = '' if m == 'twoslot' else 'none'


w_mode.observe(_toggle, names='value')
w_go.on_click(_doc)
_toggle()

display(W.VBox([
    W.HTML('<h3 style="margin:4px 0">VoxCPM2 - thêm nhấn nhá / ngữ điệu</h3>'
           '<div style="color:#666;font-size:13px;margin-bottom:8px">'
           'Giọng đều đều là vì <b>Ultimate copy ngữ điệu từ file mẫu</b> (mẫu phẳng -> ra phẳng). '
           'Thử: <b>hạ cfg_value về 1.3-1.6</b> (làm ngay được); hoặc <b>Ghép 2 giọng</b> '
           '(upload thêm 1 clip đọc diễn cảm làm nguồn ngữ điệu, giữ chất giọng của bạn); '
           'hoặc <b>Style</b> + chỉ thị tiếng Anh.</div>'),
    w_mode, w_dir, w_pros_file, w_pros_text,
    W.HTML('<div style="height:6px"></div>'),
    w_cfg, w_steps, w_max, w_gap_s, w_gap_p, w_text, w_go, w_out,
]))
